In [ ]:
import os
import re
import json
import random
from pathlib import Path
from typing import List, Tuple

import cv2
import torch
from PIL import Image
from tqdm import tqdm
from torchvision import transforms


ACTIONS = [
    "boxing",
    "handclapping",
    "handwaving",
    "jogging",
    "running",
    "walking",
]

ACTION_TO_ID = {
    action: idx for idx, action in enumerate(ACTIONS)
}

ID_TO_ACTION = {
    idx: action for action, idx in ACTION_TO_ID.items()
}

In [ ]:
def parse_subject_id(path: Path) -> int:
    match = re.search(r"person(\d+)", path.name.lower())

    if match is None:
        raise ValueError(f"Cannot parse subject id from filename: {path.name}")

    return int(match.group(1))


def parse_action(path: Path) -> str:
    lower = path.name.lower()

    for action in ACTIONS:
        if action in lower:
            return action

    parent = path.parent.name.lower()

    if parent in ACTION_TO_ID:
        return parent

    raise ValueError(f"Cannot parse action label from path: {path}")


def find_videos(root: str) -> List[Path]:
    root = Path(root)
    videos = sorted(list(root.rglob("*.avi")))

    if len(videos) == 0:
        raise RuntimeError(f"No .avi files found under {root}")

    return videos


def read_video_cv2(
    path: Path,
    resize_hw: Tuple[int, int] = (72, 72)
) -> List[Image.Image]:
    cap = cv2.VideoCapture(str(path))
    frames = []

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")

    while True:
        ok, frame = cap.read()

        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(
            gray,
            resize_hw,
            interpolation=cv2.INTER_AREA
        )

        rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        frames.append(Image.fromarray(rgb))

    cap.release()

    if len(frames) == 0:
        raise RuntimeError(f"No frames decoded from: {path}")

    return frames

In [ ]:
def make_spatial_transform(
    crop_hw: Tuple[int, int] = (64, 64),
    augment: bool = False,
):
    if augment:
        return transforms.Compose([
            transforms.RandomCrop(crop_hw),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=8),
            transforms.ColorJitter(
                brightness=0.15,
                contrast=0.15,
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5],
            ),
        ])

    return transforms.Compose([
        transforms.CenterCrop(crop_hw),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5],
        ),
    ])

In [ ]:
def sample_temporal_indices(
    num_frames: int,
    seq_len: int = 10,
    base_start: int = 0,
    temporal_stride: int = 1,
    reverse: bool = False,
):
    indices = [
        base_start + i * temporal_stride
        for i in range(seq_len)
    ]

    indices = [
        min(idx, num_frames - 1)
        for idx in indices
    ]

    if reverse:
        indices = indices[::-1]

    return indices

In [ ]:
def preprocess_kth_to_files_with_augmentation(
    raw_root: str,
    output_root: str,
    seq_len: int = 10,
    stride: int = 5,
    resize_hw: Tuple[int, int] = (72, 72),
    crop_hw: Tuple[int, int] = (64, 64),
    augment_train: bool = True,
    augment_copies: int = 2,
    temporal_strides: Tuple[int, ...] = (1, 2),
    reverse_prob: float = 0.0,
):
    """
    Saves preprocessed KTH subsequences as .pt files.

    Output tensor shape:
        frames: (seq_len, 3, 64, 64)

    """

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    base_transform = make_spatial_transform(
        crop_hw=crop_hw,
        augment=False,
    )

    aug_transform = make_spatial_transform(
        crop_hw=crop_hw,
        augment=True,
    )

    videos = find_videos(raw_root)

    metadata = []
    sample_idx = 0

    print(f"Found {len(videos)} videos")

    for video_path in tqdm(videos):
        try:
            action = parse_action(video_path)
            label = ACTION_TO_ID[action]
            subject_id = parse_subject_id(video_path)

            frames = read_video_cv2(
                path=video_path,
                resize_hw=resize_hw,
            )

            num_frames = len(frames)

            max_temporal_stride = max(temporal_strides)
            min_required_frames = (seq_len - 1) * max_temporal_stride + 1

            if num_frames < seq_len:
                continue

            valid_last_start = num_frames - seq_len

            for base_start in range(0, valid_last_start + 1, stride):

                versions = [("clean", base_transform, 1, False)]

                if augment_train:
                    for aug_id in range(augment_copies):
                        sampled_temporal_stride = random.choice(temporal_strides)

                        reverse = random.random() < reverse_prob

                        versions.append((
                            f"aug_{aug_id}",
                            aug_transform,
                            sampled_temporal_stride,
                            reverse,
                        ))

                for version_name, transform, temporal_stride, reverse in versions:

                    max_start = num_frames - ((seq_len - 1) * temporal_stride + 1)

                    if max_start < 0:
                        continue

                    if version_name == "clean":
                        start = min(base_start, max_start)
                    else:
                        offset = random.randint(0, max(stride - 1, 0))
                        start = min(base_start + offset, max_start)

                    frame_indices = sample_temporal_indices(
                        num_frames=num_frames,
                        seq_len=seq_len,
                        base_start=start,
                        temporal_stride=temporal_stride,
                        reverse=reverse,
                    )

                    clip = [
                        frames[idx]
                        for idx in frame_indices
                    ]

                    clip_tensor = torch.stack(
                        [transform(frame) for frame in clip],
                        dim=0,
                    )

                    save_name = f"sample_{sample_idx:07d}.pt"
                    save_path = output_root / save_name

                    sample = {
                        "frames": clip_tensor,
                        "label": label,
                        "action": action,
                        "subject_id": subject_id,
                        "video_path": str(video_path),
                        "start_frame": start,
                        "frame_indices": frame_indices,
                        "seq_len": seq_len,
                        "temporal_stride": temporal_stride,
                        "reversed": reverse,
                        "augmentation": version_name,
                    }

                    torch.save(sample, save_path)

                    metadata.append({
                        "file": save_name,
                        "label": label,
                        "action": action,
                        "subject_id": subject_id,
                        "video_path": str(video_path),
                        "start_frame": start,
                        "frame_indices": frame_indices,
                        "seq_len": seq_len,
                        "temporal_stride": temporal_stride,
                        "reversed": reverse,
                        "augmentation": version_name,
                    })

                    sample_idx += 1

        except Exception as e:
            print(f"Skipping {video_path}: {e}")

    metadata_path = output_root / "metadata.json"

    with open(metadata_path, "w") as f:
        json.dump(
            {
                "actions": ACTIONS,
                "action_to_id": ACTION_TO_ID,
                "id_to_action": ID_TO_ACTION,
                "num_samples": sample_idx,
                "seq_len": seq_len,
                "stride": stride,
                "resize_hw": resize_hw,
                "crop_hw": crop_hw,
                "augment_train": augment_train,
                "augment_copies": augment_copies,
                "temporal_strides": temporal_strides,
                "reverse_prob": reverse_prob,
                "samples": metadata,
            },
            f,
            indent=2,
        )

    print(f"Saved {sample_idx} samples to: {output_root}")
    print(f"Saved metadata to: {metadata_path}")

In [ ]:
preprocess_kth_to_files_with_augmentation(
    raw_root="KTH",
    output_root="KTH_preprocessed_augmented",
    seq_len=10,
    stride=5,
    resize_hw=(72, 72),
    crop_hw=(64, 64),
    augment_train=True,
    augment_copies=2,
    temporal_strides=(1, 2),
    reverse_prob=0.0,
)

In [ ]:
sample = torch.load(
    "KTH_preprocessed_augmented/sample_0000000.pt",
    weights_only=False,
)

print(sample["frames"].shape)
print(sample["label"])
print(sample["action"])
print(sample["frame_indices"])
print(sample["augmentation"])
print(sample["temporal_stride"])
print(sample["reversed"])